# Know Your Village

Explore population, distances to services, livestock and village survey answers, and find linked micro-watersheds.

Run the cells in order. Change the place, identifier or columns to explore other records. Downloads from GeoLibre use your selected tehsil; these templates start with Hilsa, Nalanda, Bihar.


## Set up Python

Run the collapsed setup cells. They import the libraries and define `read_json`, a small response reader. It reads JSON text, treats non-standard `NaN` and `Infinity` numbers as missing, and also accepts JSON returned inside a string. HTTP errors and malformed responses remain visible. Expand the cells to read the code.


In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["geopandas", "matplotlib", "requests", "pyodide-http"])
    import pyodide_http
    pyodide_http.patch_all()

import os
import re
import ast
import json
from getpass import getpass
from inspect import isawaitable
from urllib.parse import urljoin
import requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, FileLink
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})


In [ ]:
SCOPE = json.loads("{\"state\": \"Bihar\", \"district\": \"Nalanda\", \"tehsil\": \"Hilsa\"}")
API_URL = 'https://geoserver.core-stack.org/api/v1/'
STAC_URL = 'https://spatio-temporal-asset-catalog.s3.ap-south-1.amazonaws.com/CorestackCatalogs_merged_collection/tehsil_wise/catalog.json'
YEARS = list(range(2017, 2025))


In [ ]:
"""Small response reader embedded in the notebooks' collapsed setup cell."""
import json


def read_json(response):
    """Read JSON text; represent non-standard NaN/Infinity values as missing."""
    response.raise_for_status()
    raw_text = response.text.lstrip("\ufeff")
    try:
        # Some API tables contain bare NaN or Infinity, which are not JSON numbers.
        data = json.loads(raw_text, parse_constant=lambda value: None)
        # Also accept a JSON document returned as a JSON-encoded string.
        if isinstance(data, str):
            data = json.loads(data.lstrip("\ufeff"), parse_constant=lambda value: None)
        return data
    except ValueError as error:
        raise ValueError(
            "The server response is not readable JSON. "
            "Inspect response.status_code and response.text[:500], then retry the request."
        ) from error


## Choose the place and set your API key

Edit `SCOPE` in the setup cell to change the place. The [public API guide](https://docs.core-stack.org/use-precomputed-data/public-apis/) explains registration and API keys. This cell reuses `CORE_STACK_API_KEY` or asks for it privately, then stores it in this kernel’s environment. The key is sent only to the API, in the `X-API-Key` header. Restart the kernel and run from the top after changing places.


In [ ]:
place = {key: re.sub(r"[\s_]+", "_", SCOPE[key].replace("(", "").replace(")", "")).strip("_").lower()
         for key in ["state", "district", "tehsil"]}
state, district, tehsil = place["state"], place["district"], place["tehsil"]
api_key = os.environ.get("CORE_STACK_API_KEY", "").strip()
if not api_key:
    api_key = getpass("CoRE Stack API key: ")
    if isawaitable(api_key):
        api_key = await api_key
os.environ["CORE_STACK_API_KEY"] = str(api_key).strip()
api_headers = {"X-API-Key": os.environ["CORE_STACK_API_KEY"]}
display(place)


## Read the tehsil and choose a village

One request returns the tehsil’s tables. The cell keeps the tables used here, lists identifiers and shows the first record’s first ten fields. Change the selected identifier, then rerun the following cells. Blank fields mean the source did not supply a value.


In [ ]:
response = requests.get(API_URL + "get_tehsil_data/", params=place, headers=api_headers, timeout=180)
api_data = read_json(response)
required_tables = ['social_economic_indicator', 'facilities_proximity', 'livestock', 'antyodaya', 'mws_intersect_villages']
tables = {name: pd.DataFrame(api_data.get(name, [])) for name in required_tables}
display(pd.DataFrame({"Table": required_tables, "Rows": [len(tables[name]) for name in required_tables]}))
villages = tables["social_economic_indicator"]
villages = villages.loc[villages["village_id"].notna() & (villages["village_id"] != 0)]  # Exclude unassigned IDs.
display(villages[["village_name", "village_id"]])
village_id = str(villages.iloc[0]["village_id"])  # Choose another ID from the list.
village = villages.loc[villages["village_id"].astype(str) == village_id].iloc[0]
display(village.to_frame("Recorded value"))


## Discover data and descriptions in STAC

STAC lists published datasets, field descriptions, downloads and styles. Change `dataset` to another item from the collection. Asset links are used as published, wherever the files are hosted. STAC describes asset fields; API tables may use different names and units, which are shown explicitly in the examples below.


In [ ]:
collection_url = urljoin(STAC_URL, f"{state}/{district}/{tehsil}/collection.json")
response = requests.get(collection_url, timeout=90)
collection = read_json(response)
items = pd.DataFrame([{"Item": link["href"].split("/")[-1].removesuffix(".json"),
                       "URL": urljoin(collection_url, link["href"])}
                      for link in collection["links"] if link["rel"] == "item"], columns=["Item", "URL"])
# Follow a relevant item link from the collection.
dataset = "admin_boundaries_vector"
matches = items.loc[items["Item"].str.endswith("_" + dataset)]
item = None
if not matches.empty:
    item_url = matches.iloc[0]["URL"]
    response = requests.get(item_url, timeout=90)
    item = read_json(response)
    display(Markdown(item["properties"].get("description", "No description published.")))
    field_notes = pd.DataFrame(item["properties"].get("table:columns", []))
    display(field_notes.reindex(columns=["name", "type", "description"]).head(12))
    print("Published field count:", len(field_notes), "— use field_notes to see them all.")
    display(pd.DataFrame(item["assets"]).T.reindex(columns=["title", "type", "href"]))
else:
    print("This dataset is not listed in the tehsil's STAC collection. Available items:")
    display(items)


## Population and literacy

Population, Scheduled Caste and Scheduled Tribe values are counts. Literacy is a percentage. Keep the different units visible.


In [ ]:
population_fields = {"total_population_count": "Population", "total_sc_population_count": "Scheduled Caste population", "total_st_population_count": "Scheduled Tribe population", "literacy_rate_percent": "Literacy (%)"}
display(village.reindex(population_fields).rename(index=population_fields).to_frame("Recorded value"))


## How far away are services?

Compare the reported distances in kilometres. These fields give distances, not compass directions. Each point labels a service category; missing distances stay blank in the table and are omitted from the plot.


In [ ]:
facilities = tables["facilities_proximity"]
facility = facilities.loc[facilities["village_id"].astype(str) == village_id]
categories = {"essential_education": "Essential education", "higher_education": "Higher education", "essential_health": "Essential health", "advanced_health": "Advanced health", "essential_services": "Essential services", "financial_inclusion": "Financial services", "apmc_markets": "Agricultural markets", "post_harvest": "Post-harvest services", "cooperative": "Cooperatives", "livestock": "Livestock services", "agri_support_infra": "Agricultural support"}
facility = facility.iloc[0] if not facility.empty else pd.Series(dtype=object)
distances = pd.DataFrame({"Service": list(categories.values()),
                          "Distance (km)": [facility.get(f"{name}_cat_distance_in_km") for name in categories],
                          "Facility": [facility.get(f"{name}_facility_label") for name in categories]}).set_index("Service")
distances["Distance (km)"] = pd.to_numeric(distances["Distance (km)"], errors="coerce")
display(distances)
available = distances["Distance (km)"].dropna().sort_values()
if not available.empty:
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hlines(available.index, 0, available, color="#b3d9d3", linewidth=3)
    ax.scatter(available, available.index, color="#227b71", s=50)
    for label, value in available.items():
        ax.annotate(f"{value:g} km", (value, label), xytext=(5, 0), textcoords="offset points", va="center")
    ax.set(xlabel="Distance (km)", title=f"Access to services · {village['village_name']}")
    ax.set_xlim(left=0, right=available.max() * 1.2 if available.max() > 0 else 1)
    plt.tight_layout()
    plt.show()
else:
    print("No service distances were returned for this village.")


## Livestock counts

Show animal types separately. The all-livestock total is a reference value, not another category to add to the chart.


In [ ]:
livestock = tables["livestock"]
records = livestock.loc[livestock["village_id"].astype(str) == village_id]
if not records.empty:
    animals = records.iloc[0]
    fields = {"cattle_total": "Cattle", "buffalo_total": "Buffalo", "sheep_total": "Sheep", "goat_total": "Goats", "pig_total": "Pigs"}
    counts = pd.to_numeric(animals.reindex(fields), errors="coerce").rename(index=fields)
    display(animals.reindex(["all_livestock_total", *fields]).to_frame("Count"))
    counts.dropna().plot.barh(figsize=(7, 3), xlabel="Animals", color="#a47b48", title="Livestock by type")
    plt.tight_layout()
    plt.show()
else:
    print("No livestock record was returned for this village.")


## Explore a village survey topic

The collapsed reference lists the original Mission Antyodaya fields under the village report’s headings. Change `group` in the next cell to read a different topic. The category value is shown beside its source answers; values from different topics are not assumed to share a scale.


In [ ]:
survey_groups = {'road_connectivity': [{'col': 'is_village_connected_to_all_weather_road',
                        'label': 'All-weather road connection',
                        'repr': 'binary'},
                       {'col': 'availability_of_internal_pucca_road',
                        'label': 'Internal pucca road quality',
                        'repr': 'string'},
                       {'col': 'availability_of_public_transport',
                        'label': 'Public transport availability',
                        'repr': 'string'},
                       {'col': 'availability_of_railway_station',
                        'label': 'Railway station availability',
                        'repr': 'binary'}],
 'energy_access': [{'col': 'availablility_hours_of_domestic_electricity',
                    'label': 'Domestic electricity supply (hours/day)',
                    'repr': 'string'},
                   {'col': 'availability_of_elect_supply_to_msme',
                    'label': 'Electricity supply to MSME units',
                    'repr': 'binary'},
                   {'col': 'total_hhd', 'label': 'Total number of households', 'repr': 'numeric'},
                   {'col': 'total_hhd_with_clean_energy',
                    'label': 'HHs using clean energy (LPG / Biogas)',
                    'repr': 'numeric'}],
 'housing_quality': [{'col': 'total_hhd', 'label': 'Total number of households', 'repr': 'numeric'},
                     {'col': 'total_hhd_with_kuccha_wall_kuccha_roof',
                      'label': 'HHs with kuccha wall & kuccha roof',
                      'repr': 'numeric'},
                     {'col': 'total_hhd_got_benefit_under_state_housing_scheme',
                      'label': 'State housing scheme beneficiaries',
                      'repr': 'numeric'},
                     {'col': 'total_hhd_have_got_pmay_house',
                      'label': 'PMAY houses (completed / sanctioned)',
                      'repr': 'numeric'},
                     {'col': 'total_hhd_in_pmay_permanent_wait_list',
                      'label': 'PMAY permanent waitlist households',
                      'repr': 'numeric'},
                     {'col': 'total_hhd_availing_pmuy_benefits',
                      'label': 'PMUY (Ujjwala Yojana) beneficiaries',
                      'repr': 'numeric'}],
 'maternal_child_health': [{'col': 'availability_of_mother_child_health_facilities',
                            'label': 'Availability of Mother and Child Health facilities',
                            'repr': 'binary'},
                           {'col': 'is_aanganwadi_centre_available',
                            'label': 'Availability of Aanganwadi Centre',
                            'repr': 'binary'},
                           {'col': 'is_early_childhood_edu_provided_in_anganwadi',
                            'label': 'Is Early Childhood Education provided in the Anganwadi',
                            'repr': 'binary'},
                           {'col': 'total_childs_aged_0_to_3_years',
                            'label': 'Total no of children in the age group of 0-3 years',
                            'repr': 'numeric'},
                           {'col': 'total_childs_aged_0_to_3_years_reg_under_aanganwadi',
                            'label': 'Total no of children aged 0-3 years registered in Aanganwadi',
                            'repr': 'numeric'},
                           {'col': 'total_no_of_pregnant_women',
                            'label': 'Total number of Pregnant women',
                            'repr': 'numeric'},
                           {'col': 'total_no_of_pregnant_women_receiving_services_under_icds',
                            'label': 'No of pregnant women receiving services under ICDS',
                            'repr': 'numeric'},
                           {'col': 'total_no_of_lactating_mothers',
                            'label': 'Total number of lactating mothers',
                            'repr': 'numeric'},
                           {'col': 'total_anemic_pregnant_women',
                            'label': 'No. of Anaemic Pregnant Women',
                            'repr': 'numeric'},
                           {'col': 'total_childs_aged_0_to_3_years_immunized',
                            'label': 'No of children aged 0-3 years immunized',
                            'repr': 'numeric'},
                           {'col': 'total_no_of_newly_born_children',
                            'label': 'Total number of newly born children during the year',
                            'repr': 'numeric'},
                           {'col': 'total_no_of_newly_born_underweight_children',
                            'label': 'No of newly born children underweight',
                            'repr': 'numeric'},
                           {'col': 'gp_total_no_of_beneficiaries_receiving_benefits_under_pmjay',
                            'label': 'No. of beneficiaries receiving benefits under PMJAY',
                            'repr': 'numeric'},
                           {'col': 'gp_total_no_of_eligible_beneficiaries_under_pmjay',
                            'label': 'Total no. of eligible beneficiaries under PMJAY',
                            'repr': 'numeric'},
                           {'col': 'total_hhd_registered_under_pmjay',
                            'label': 'No. of Households registered under PMJAY/State Health Insurance',
                            'repr': 'numeric'},
                           {'col': 'total_no_of_beneficiaries_receiving_benefits_under_pmmvy',
                            'label': 'No of beneficiaries receiving benefits under PMMVY',
                            'repr': 'numeric'},
                           {'col': 'total_no_of_eligible_beneficiaries_under_pmmvy',
                            'label': 'Total no of eligible beneficiaries under PMMVY',
                            'repr': 'numeric'}],
 'water_sanitation': [{'col': 'availability_of_piped_tap_water',
                       'label': 'Availability of Piped tap water (Coverage)',
                       'repr': 'string'},
                      {'col': 'total_hhd', 'label': 'Total number of households', 'repr': 'numeric'},
                      {'col': 'total_hhd_having_piped_water_connection',
                       'label': 'No of households having piped water connection',
                       'repr': 'numeric'},
                      {'col': 'total_hhd_not_having_sanitary_latrines',
                       'label': 'No of households not having sanitary latrines',
                       'repr': 'numeric'},
                      {'col': 'availability_of_drainage_system',
                       'label': 'Availability of drainage facilities',
                       'repr': 'string'},
                      {'col': 'is_community_waste_disposal_system',
                       'label': 'Community waste disposal system',
                       'repr': 'binary'},
                      {'col': 'is_community_biogas_waste_recycle_for_production',
                       'label': 'Community bio gas or recycle of waste',
                       'repr': 'binary'}],
 'financial_inclusion': [{'col': 'is_bank_available', 'label': 'Availability of banks', 'repr': 'binary'},
                         {'col': 'is_atm_available', 'label': 'Availability of ATM', 'repr': 'binary'},
                         {'col': 'is_bank_buss_correspondent_with_internet',
                          'label': 'Availability of Business Correspondent with internet connectivity',
                          'repr': 'binary'},
                         {'col': 'total_shg',
                          'label': 'Number of Self Help Groups (SHGs)',
                          'repr': 'numeric'},
                         {'col': 'total_shg_accessed_bank_loans',
                          'label': 'No of SHGs which accessed bank loans',
                          'repr': 'numeric'},
                         {'col': 'total_hhd', 'label': 'Total number of households', 'repr': 'numeric'},
                         {'col': 'total_hhd_availing_pmjdy_bank_ac',
                          'label': 'Number of households having Jan-Dhan bank account',
                          'repr': 'numeric'}],
 'social_protection': [{'col': 'gp_total_hhd_eligible_under_nfsa',
                        'label': 'Total number of eligible households under NFSA',
                        'repr': 'numeric'},
                       {'col': 'gp_total_hhd_receiving_food_grains_from_fps',
                        'label': 'Total no of households receiving food grains from Fair Price Shops',
                        'repr': 'numeric'},
                       {'col': 'total_hhd', 'label': 'Total households', 'repr': 'numeric'},
                       {'col': 'total_hhd_having_bpl_cards',
                        'label': 'Number of Households having BPL ration cards',
                        'repr': 'numeric'},
                       {'col': 'total_hhd_availing_pension_under_nsap',
                        'label': 'Number of Households getting pensions under NSAP',
                        'repr': 'numeric'}],
 'institutionalization': [{'col': 'total_hhd', 'label': 'Total number of households', 'repr': 'numeric'},
                          {'col': 'total_hhd_mobilized_into_shg',
                           'label': 'Number of households mobilized into SHGs',
                           'repr': 'numeric'},
                          {'col': 'total_no_of_shg_promoted',
                           'label': 'Number of SHGs federated into Village Organisations',
                           'repr': 'numeric'},
                          {'col': 'total_shg',
                           'label': 'Number of Self Help Groups (SHGs)',
                           'repr': 'numeric'},
                          {'col': 'total_hhd_mobilized_into_pg',
                           'label': 'Number of households mobilized into Producer Groups',
                           'repr': 'numeric'},
                          {'col': 'availability_of_fpos_pacs',
                           'label': 'Availability of Farmers Collective (Farmer Producer Organizations '
                                    '(FPOs)/Primary Agricultural Credit Societies (PACS))',
                           'repr': 'string'}],
 'civic_infrastructure': [{'col': 'availability_of_panchayat_bhawan',
                           'label': 'Availability of Panchayat Bhawan',
                           'repr': 'binary'},
                          {'col': 'is_post_office_available',
                           'label': 'Availability of Post office/Sub-Post office',
                           'repr': 'binary'},
                          {'col': 'total_no_of_elected_representatives',
                           'label': 'Total no of elected representatives',
                           'repr': 'numeric'},
                          {'col': 'total_no_of_elect_rep_undergone_training_under_rgsa',
                           'label': 'No of elected representatives undergone refresher training under RGSA',
                           'repr': 'numeric'},
                          {'col': 'total_no_of_elect_rep_oriented_under_rgsa',
                           'label': 'No of elected representatives oriented under RGSA',
                           'repr': 'numeric'},
                          {'col': 'availability_of_public_information_board',
                           'label': "Availability of Public Information Board under People's Plan Campaign",
                           'repr': 'string'},
                          {'col': 'availability_of_public_library',
                           'label': 'Availability of Public Library',
                           'repr': 'binary'}],
 'livelihoods_employment': [{'col': 'total_hhd', 'label': 'Total number of households', 'repr': 'numeric'},
                            {'col': 'total_hhd_engaged_in_farm_activities',
                             'label': 'Number of households engaged majorly in farm activities',
                             'repr': 'numeric'}],
 'livelihoods_forest_resources': [{'col': 'availability_of_community_forest',
                                   'label': 'Availability of Community Forest',
                                   'repr': 'binary'},
                                  {'col': 'availability_of_minor_forest_production',
                                   'label': 'Availability of minor forest production',
                                   'repr': 'binary'},
                                  {'col': 'total_hhd',
                                   'label': 'Total number of households',
                                   'repr': 'numeric'},
                                  {'col': 'total_hhd_source_of_minor_forest_production',
                                   'label': 'Number of Households where only source of livelihood is minor '
                                            'forest production',
                                   'repr': 'numeric'}],
 'livelihoods_fisheries': [{'col': 'availability_of_aquaculture_ext_facility',
                            'label': 'Extension facilities for Aquaculture',
                            'repr': 'binary'},
                           {'col': 'availability_of_fish_community_ponds',
                            'label': 'Community Ponds Used for Fisheries',
                            'repr': 'binary'},
                           {'col': 'availability_of_fish_farming',
                            'label': 'Pisciculture - InLand Fishery/Coastal Fishery',
                            'repr': 'binary'}],
 'livelihoods_alternative_farming': [{'col': 'is_bee_farming', 'label': 'Bee Keeping', 'repr': 'binary'},
                                     {'col': 'is_sericulture',
                                      'label': 'Sericulture (Silk Production)',
                                      'repr': 'binary'}],
 'livelihoods_cottage_traditional_industry': [{'col': 'availability_of_cottage_small_scale_units',
                                               'label': 'Availability of cottage and small scale units',
                                               'repr': 'binary'},
                                              {'col': 'total_hhd',
                                               'label': 'Total number of households',
                                               'repr': 'numeric'},
                                              {'col': 'total_hhd_engaged_cottage_small_scale_units',
                                               'label': 'Number of Households engaged in cottage/small scale '
                                                        'units',
                                               'repr': 'numeric'},
                                              {'col': 'is_handloom', 'label': 'Handloom', 'repr': 'binary'},
                                              {'col': 'is_handicrafts',
                                               'label': 'Handicrafts',
                                               'repr': 'binary'}],
 'livelihoods_common_resources': [{'col': 'is_common_pastures_available',
                                   'label': 'Common pastures as per revenue records',
                                   'repr': 'binary'}],
 'livestock_veterinary': [{'col': 'availability_of_livestock_extension_services',
                           'label': 'Availability of Livestock Extension services',
                           'repr': 'string'},
                          {'col': 'is_veterinary_hospital_available',
                           'label': 'Availability of Veterinary Clinic or Hospital',
                           'repr': 'binary'},
                          {'col': 'availability_of_goatary_dev_project',
                           'label': 'Project supporting Goatary Development',
                           'repr': 'binary'},
                          {'col': 'availability_of_pigery_development',
                           'label': 'Project supporting Piggery Development',
                           'repr': 'binary'},
                          {'col': 'availability_of_poultry_dev_project',
                           'label': 'Project supporting Poultry Development',
                           'repr': 'binary'},
                          {'col': 'availability_of_milk_routes',
                           'label': 'Availability of Milk Collection Centre/Milk routes/Chilling Centres',
                           'repr': 'binary'}],
 'agriculture_land_cultivation': [{'col': 'area_irrigated_in_hac',
                                   'label': 'Total area irrigated (ha)',
                                   'repr': 'numeric'},
                                  {'col': 'net_sown_area_in_hac',
                                   'label': 'Net sown Area (ha)',
                                   'repr': 'numeric'},
                                  {'col': 'net_sown_area_kharif_in_hac',
                                   'label': 'Net sown Area during Kharif season (ha)',
                                   'repr': 'numeric'},
                                  {'col': 'net_sown_area_other_in_hac',
                                   'label': 'Net sown Area during other seasons (ha)',
                                   'repr': 'numeric'},
                                  {'col': 'net_sown_area_rabi_in_hac',
                                   'label': 'Net sown Area during Rabi season (ha)',
                                   'repr': 'numeric'},
                                  {'col': 'total_cultivable_area_in_hac',
                                   'label': 'Total Cultivable Area (ha)',
                                   'repr': 'numeric'}],
 'agriculture_irrigation_watershed': [{'col': 'availability_of_major_source_of_irrigation',
                                       'label': 'Main Source of irrigation',
                                       'repr': 'string'},
                                      {'col': 'availability_of_rain_harvest_system',
                                       'label': 'Availability of Community Rain Water Harvesting '
                                                'System/Pond/Dam/Check Dam',
                                       'repr': 'binary'},
                                      {'col': 'availability_of_watershed_dev_project',
                                       'label': 'Whether village is part of Watershed Development Project',
                                       'repr': 'binary'},
                                      {'col': 'total_approved_labour_budget_for_year',
                                       'label': 'Total approved Labour Budget for the year (₹)',
                                       'repr': 'numeric'},
                                      {'col': 'total_expenditure_approved_under_nrm_labour_budget_during_yr',
                                       'label': 'Total expenditure approved under NRM in the Labour Budget '
                                                '(₹)',
                                       'repr': 'numeric'},
                                      {'col': 'no_of_farmers_using_drip_sprinkler',
                                       'label': 'Number of farmers using drip/sprinkler irrigation',
                                       'repr': 'numeric'},
                                      {'col': 'total_no_of_farmers',
                                       'label': 'Total no of farmers',
                                       'repr': 'numeric'}],
 'agriculture_support_services': [{'col': 'is_fertilizer_shop_available',
                                   'label': 'Availability of fertilizer shop',
                                   'repr': 'binary'},
                                  {'col': 'is_govt_seed_centre_available',
                                   'label': 'Availability of government seed centres',
                                   'repr': 'binary'},
                                  {'col': 'is_soil_testing_centre_available',
                                   'label': 'Availability of soil testing centres',
                                   'repr': 'binary'},
                                  {'col': 'total_no_of_farmers',
                                   'label': 'Total no of farmers',
                                   'repr': 'numeric'},
                                  {'col': 'total_no_of_farmers_received_benefit_under_pmfby',
                                   'label': 'No of farmers received benefits under PMFBY',
                                   'repr': 'numeric'},
                                  {'col': 'total_no_of_farmers_registered_under_pmkpy',
                                   'label': 'Total number of farmers registered under PM Kisan Pension '
                                            'Yojana',
                                   'repr': 'numeric'},
                                  {'col': 'total_no_of_farmers_add_fert_in_soil_as_per_report',
                                   'label': 'Number of farmers received the soil testing report',
                                   'repr': 'numeric'}],
 'agricultural_markets': [{'col': 'availability_of_market',
                           'label': 'Availability of markets',
                           'repr': 'string'},
                          {'col': 'availability_of_food_storage_warehouse',
                           'label': 'Availability of warehouse for Food Grain Storage',
                           'repr': 'binary'}],
 'agriculture_organic_farming': [{'col': 'total_no_farmers_adopted_organic_farming',
                                  'label': 'No of farmers adopted organic farming',
                                  'repr': 'numeric'},
                                 {'col': 'total_no_of_farmers',
                                  'label': 'Total no of farmers',
                                  'repr': 'numeric'}]}


## Choose a survey topic

Try `agriculture_land_cultivation`, `agriculture_support_services`, `agricultural_markets`, `water_sanitation` or another group listed below. The same cell shows the category value and the original questions and answers.


In [ ]:
display(pd.DataFrame({"Available group": list(survey_groups)}))
group = "agriculture_land_cultivation"
survey = tables["antyodaya"]
matches = survey.loc[survey["village_id"].astype(str) == village_id]
if not matches.empty:
    answers = matches.iloc[0]
    print("Category value:", answers.get(group + "_cat_value"))
    questions = pd.DataFrame(survey_groups[group])
    questions["Answer"] = [answers.get(field) for field in questions["col"]]
    display(questions[["label", "col", "Answer"]].rename(columns={"label": "Question", "col": "API field"}))
    # Choose one unit for the plot; other answers remain in the table.
    plot_unit = "(ha)"  # Change this to a unit in the displayed question labels.
    numeric = questions.loc[(questions["repr"] == "numeric") & questions["label"].str.contains(plot_unit, regex=False)].set_index("label")["Answer"]
    numeric = pd.to_numeric(numeric, errors="coerce").dropna()
    if not numeric.empty:
        numeric.plot.barh(figsize=(9, 4), xlabel=f"Recorded value {plot_unit}", title=group.replace("_", " ").title())
        plt.tight_layout()
        plt.show()
else:
    print("No Mission Antyodaya record was returned for this village.")


## Agricultural land and access to services

Join the survey and service tables on `village_id`. Change the selected columns to combine other fields from the examples above. Source matches remain separate rows.


In [ ]:
agriculture_columns = ["village_id", "agriculture_land_cultivation_cat_value", "agricultural_markets_cat_value", "net_sown_area_in_hac"]
service_columns = ["village_id", "apmc_markets_cat_distance_in_km", "agri_support_infra_cat_distance_in_km"]
agriculture = survey.loc[survey["village_id"].astype(str) == village_id].reindex(columns=agriculture_columns).copy()
access = tables["facilities_proximity"].reindex(columns=service_columns).copy()
agriculture["village_id"], access["village_id"] = agriculture["village_id"].astype(str), access["village_id"].astype(str)
display(agriculture.merge(access, on="village_id", how="left"))


## Find the village boundary and linked MWS

The geometry API calls the village identifier `vill_ID`. Use it to select the boundary. The tehsil’s intersection table supplies the linked MWS identifiers; it does not allocate population or livestock between them.


In [ ]:
response = requests.get(API_URL + "get_village_geometries/", params=place, headers=api_headers, timeout=180)
boundaries = gpd.GeoDataFrame.from_features(read_json(response)["features"], crs="EPSG:4326")
boundary = boundaries.loc[boundaries["vill_ID"].astype(str) == village_id]
if not boundary.empty:
    boundary.plot(figsize=(5, 5), color="#d7e9f5", edgecolor="#376987")
    plt.title(str(village["village_name"]))
    plt.axis("off")
    plt.show()
links = tables["mws_intersect_villages"].copy()
links["village ids"] = links["village ids"].map(ast.literal_eval)
linked = links.explode("village ids")
display(linked.loc[linked["village ids"].astype(str) == village_id, ["mws uid", "area_in_ha"]])


## Look up administrative names from coordinates

Use a point inside the village, or supply your own latitude and longitude.


In [ ]:
if not boundary.empty:
    point = boundary.geometry.iloc[0].representative_point()
    response = requests.get(API_URL + "get_admin_details_by_latlon/", params={"latitude": point.y, "longitude": point.x}, headers=api_headers, timeout=90)
    display(read_json(response) if response.ok else {"HTTP status": response.status_code, "Response": response.text[:500]})
